# Volume 13, Chapter 13: Gowers norms and the Green–Tao theorem (Python companion)

Exercises first, solutions at the end. We compute Gowers norms on $\mathbb Z_N$ with the fast Fourier transform, test the generalized von Neumann theorem, watch a quadratic phase fool Fourier analysis, and count progressions of primes against the Green–Tao–Ziegler prediction. The computer science application: **testing whether a Boolean function is a low-degree polynomial**, where the acceptance probability of the natural test is exactly a Gowers norm. Requires `numpy`.

In [ ]:
import math, random
import numpy as np
rng = np.random.default_rng(13)

def u2_4(f):
    # ||f||_{U^2}^4 = sum |f^(xi)|^4 with f^(xi) = E_x f(x) e(-x xi/N)
    F = np.fft.fft(f) / len(f)
    return float(np.sum(np.abs(F)**4))
def delta(f, h):
    return np.roll(f, -h) * np.conj(f)          # f(x+h) * conj f(x)
def gowers(f, k):
    # ||f||_{U^k}, using the recursion ||f||_{U^k}^{2^k} = E_h ||Delta_h f||_{U^{k-1}}^{2^{k-1}}
    def power(g, k):
        if k == 1: return abs(np.mean(g))**2
        if k == 2: return u2_4(g)
        return np.mean([power(delta(g, h), k - 1) for h in range(len(g))])
    return power(f, k) ** (1 / 2**k)

## Exercises

**P13.1 (★) A table of norms.** For $N=211$, compute $\|f\|_{U^2},\|f\|_{U^3},\|f\|_{U^4}$ for random signs, the linear phase $e(\frac{5x}N)$, the quadratic phase $e(\frac{x^2}N)$, the cubic phase $e(\frac{x^3}N)$, and the balanced indicator of a random set of density $\frac12$. Check Lemma 13.2 and Exercise 13.3.

**P13.2 (★) The generalized von Neumann theorem.** For random functions $f_0,\dots,f_{k-1}$ with $|f_i|\le1$ (some of them structured), compare $|\mathbb E_{x,d}\prod_if_i(x+id)|$ with $\min_j\|f_j\|_{U^{k-1}}$ for $k=3,4$.

**P13.3 (★★) Fooling Fourier analysis.** Let $N=1009$ and $A=\{x:\ \{\frac{x^2}N\}<\frac15\}$, where $\{\cdot\}$ is the fractional part. Compute the density $\alpha$, the largest non-trivial Fourier coefficient of $\mathbf1_A$, the normalized counts of $3$-APs and $4$-APs, and compare with $\alpha^3$ and $\alpha^4$. Do the same for a random set of the same density.

**P13.4 (★★) Primes and the $W$-trick.** Compute $\frac1N\sum_{n\le N}(\Lambda(n)-1)(-1)^n$ (Exercise 13.4), then the same for $\Lambda_{b,W}$ with $W=2\cdot3\cdot5\cdot7$ and $b=1$. Then count $4$-term progressions of primes up to $10^5$ and compare with the prediction of Section 13.3.

**P13.5 (★★, computer science) Testing low-degree polynomials.** A function $f:\mathbb F_2^n\to\mathbb F_2$ is claimed to be a polynomial of degree at most $s$. The *Gowers test* picks $x,h_1,\dots,h_{s+1}$ at random and accepts if $\sum_{\omega\in\{0,1\}^{s+1}}f(x+\omega\cdot h)=0$. Show that the acceptance probability is $\frac12+\frac12\|(-1)^f\|_{U^{s+1}}^{2^{s+1}}$, compute it exactly for $n=10$, $s=2$ and several functions, and compare with a simulation of the test.

## Solutions

### P13.1

In [ ]:
N = 211
x = np.arange(N)
funcs = {"random signs": rng.choice([-1.0, 1.0], N),
         "e(5x/N)": np.exp(2j * np.pi * 5 * x / N),
         "e(x^2/N)": np.exp(2j * np.pi * x**2 / N),
         "e(x^3/N)": np.exp(2j * np.pi * (x**3 % N) / N),
         "1_A - 1/2, random A": (rng.random(N) < 0.5) - 0.5}
print(f"{'function':22s} {'U^2':>8s} {'U^3':>8s} {'U^4':>8s}")
for name, f in funcs.items():
    print(f"{name:22s} {gowers(f, 2):8.4f} {gowers(f, 3):8.4f} {gowers(f, 4):8.4f}")
print("N^(-1/4) =", round(N ** -0.25, 4))

The table follows the theory line by line. The linear phase $e(\frac{5x}N)$ has all norms equal to $1$: it is a character, the extreme structured function for $U^2$ and hence for every $U^k$ (Lemma 13.2). The quadratic phase has $\|f\|_{U^2}=0.2624=N^{-1/4}$ exactly, as Exercise 13.2 predicts, but $\|f\|_{U^3}=\|f\|_{U^4}=1$. The cubic phase is small in $U^2$ and $U^3$ and equal to $1$ in $U^4$ (Exercise 13.3 with $k=5$). The random functions have norms that grow with $k$, as Lemma 13.2 requires, but stay well below $1$; they would tend to $0$ as $N\to\infty$ for each fixed $k$, though slowly for large $k$, because the $2^k$-fold products include many coincidences at this small $N$.

### P13.2

In [ ]:
def ap_avg(fs):
    k = len(fs); N = len(fs[0]); x = np.arange(N); tot = 0
    for d in range(N):
        prod = np.ones(N, dtype=complex)
        for i, f in enumerate(fs): prod *= f[(x + i * d) % N]
        tot += prod.sum()
    return tot / N**2
N = 101; x = np.arange(N)
for k in [3, 4]:
    for trial in range(3):
        fs = []
        for i in range(k):
            kind = rng.integers(3)
            if kind == 0: f = rng.choice([-1.0, 1.0], N)
            elif kind == 1: f = np.exp(2j * np.pi * rng.integers(N) * x**2 / N)
            else: f = (rng.random(N) < 0.3) * 1.0
            fs.append(f)
        lhs = abs(ap_avg(fs)); rhs = min(gowers(f, k - 1) for f in fs)
        print(f"k={k}: |average| = {lhs:.4f} <= min_j ||f_j||_U^{k-1} = {rhs:.4f}: {lhs <= rhs + 1e-12}")

In every trial the progression average is far below the bound: the bound is at least $0.31$, while the averages are below $0.011$. The theorem is an inequality designed to be used when the right side is small; here no function is uniform enough (the indicators of random sets of density $0.3$ have mean $0.3$, hence $U^{k-1}$ norm at least $0.3$ by Lemma 13.2), and the averages are small for a different reason, the random signs. Setting all functions equal to quadratic phases with suitable powers, as in Exercise 13.3, gives averages equal to $1$ with all norms equal to $1$: the bound is sharp.

### P13.3

In [ ]:
def ap_counts(ind):
    N = len(ind); x = np.arange(N); c3 = c4 = 0
    for d in range(N):
        a = ind[x] * ind[(x + d) % N] * ind[(x + 2 * d) % N]
        c3 += a.sum(); c4 += (a * ind[(x + 3 * d) % N]).sum()
    return c3 / N**2, c4 / N**2
N = 1009; x = np.arange(N)
A = ((x * x % N) / N < 0.2).astype(float)
alpha = A.mean()
R = np.zeros(N); R[rng.choice(N, int(A.sum()), replace=False)] = 1
for name, S in [("quadratic set", A), ("random set", R)]:
    a = S.mean(); F = np.abs(np.fft.fft(S - a)) / N
    t3, t4 = ap_counts(S)
    print(f"{name}: alpha={a:.4f}, max Fourier coeff {F[1:].max():.4f}; 3-APs / alpha^3 = {t3 / a**3:.3f}; 4-APs / alpha^4 = {t4 / a**4:.3f}")

Both sets have density $0.21$ and essentially the same largest non-trivial Fourier coefficient, $0.037$ and $0.035$. The random set has ratios $1.01$ and $1.11$ to the predictions $\alpha^3$ and $\alpha^4$, the second off by the random fluctuation of a count of about $2000$ progressions. The quadratic level set has a few more $3$-APs than expected at this small $N$ (ratio $1.07$), but $1.75$ times the expected number of $4$-APs: the identity $x^2-3(x+d)^2+3(x+2d)^2-(x+3d)^2=0$ means that when three of the squares lie in $[0,\frac N5)$ modulo $N$, the fourth is pushed towards that interval too. Fourier analysis cannot see the difference between the two sets, and the $U^3$ norm can.

### P13.4

In [ ]:
M = 10**6
isp = np.ones(M + 1, bool); isp[:2] = False
for i in range(2, int(M**0.5) + 1):
    if isp[i]: isp[i*i::i] = False
Lam = np.zeros(M + 1)
for p in np.nonzero(isp)[0]:
    q = p
    while q <= M: Lam[q] = math.log(p); q *= p
n = np.arange(1, M + 1)
print("(1/N) sum (Lambda(n)-1)(-1)^n, N=1e6:", round(float(np.mean((Lam[1:] - 1) * (-1.0)**n)), 4))
W, b = 210, 1
Nw = (M - b) // W
LW = (48 / 210) * Lam[W * np.arange(1, Nw + 1) + b]          # phi(210) = 48
m = np.arange(1, Nw + 1)
print(f"W-tricked, N={Nw}: mean {LW.mean():.4f}, (1/N) sum (Lambda_bW - 1)(-1)^n = {np.mean((LW - 1) * (-1.0)**m):.4f}, "
      f"mod-3 averages {[round(float(LW[m % 3 == c].mean()), 3) for c in range(3)]}")
# 4-term progressions of primes up to 1e5
N = 10**5; k = 4; c = 0
for d in range(1, (N - 2) // (k - 1) + 1):
    mask = isp[:N - (k - 1) * d + 1].copy()
    for i in range(1, k): mask &= isp[i * d:N - (k - 1 - i) * d + 1]
    c += int(mask.sum())
S4 = 1.0
for p in np.nonzero(isp[:10**6])[0]:
    S4 *= (p - 1) * (1 + max(p - k, 0)) / p**2 / (1 - 1 / p)**k
h = 250
xs = np.arange(2 + h / 2, N, h); ds = np.arange(1 + h / 2, N / (k - 1), h)
X, D = np.meshgrid(xs, ds, indexing="ij")
val = np.prod([np.log(X + i * D) for i in range(k)], axis=0)
pred = S4 * np.sum(np.where(X + (k - 1) * D <= N, 1 / val, 0)) * h * h
print(f"4-APs of primes up to 1e5: {c}; S_4 = {S4:.5f}; prediction {pred:.0f}; ratio {c / pred:.3f}")

The average of $(\Lambda(n)-1)(-1)^n$ is $-0.9996$ at $N=10^6$, as in Exercise 13.4: the primes are odd, and this one character carries almost all of the bias. After the $W$-trick with $W=210$, $\Lambda_{210,1}$ has mean $1.0001$, its correlation with $(-1)^n$ drops to $-0.016$, and its averages over the classes modulo $3$ are $1.02$, $1.00$ and $0.98$, consistent with no bias (Lemma 13.8) up to the fluctuations of a sum over $4761$ terms.

The $389\,606$ progressions of four primes up to $10^5$ are $0.986$ times the Green–Tao–Ziegler prediction with $\mathfrak S_4=2.85825$, the value of Figure 13.2. The agreement improves slowly with $N$, as the figure shows.

### P13.5

In [ ]:
n = 10; Nn = 2**n
X = np.arange(Nn)
def popcount_parity(v):
    v = v.copy(); p = np.zeros_like(v)
    while v.any(): p ^= v & 1; v >>= 1
    return p
bits = [(X >> i) & 1 for i in range(n)]
def wht(F):
    F = F.astype(float).copy(); h = 1
    while h < len(F):
        F = F.reshape(-1, 2 * h); a, b = F[:, :h].copy(), F[:, h:].copy()
        F[:, :h], F[:, h:] = a + b, a - b; F = F.reshape(-1); h *= 2
    return F / len(F)
def U_pow(F, k):
    # ||F||_{U^k}^{2^k} on F_2^n for real F, with Delta_h F(x) = F(x+h) F(x)
    if k == 2: return float(np.sum(wht(F)**4))
    return float(np.mean([U_pow(F[X ^ h] * F, k - 1) for h in range(Nn)]))
def test_sim(f, s, trials=20000):
    acc = 0
    for _ in range(trials):
        x = rng.integers(Nn); hs = rng.integers(Nn, size=s + 1); tot = 0
        for w in range(2**(s + 1)):
            y = x
            for i in range(s + 1):
                if (w >> i) & 1: y ^= hs[i]
            tot ^= f[y]
        acc += (tot == 0)
    return acc / trials
quad = (bits[0] & bits[1]) ^ (bits[2] & bits[5]) ^ (bits[3] & bits[7] ) ^ bits[4]
noisy = quad ^ (rng.random(Nn) < 0.05)
cubic = (bits[0] & bits[1] & bits[2]) ^ (bits[3] & bits[4])
maj = (sum(bits) > n // 2).astype(int)
rand = rng.integers(2, size=Nn)
s = 2
for name, f in [("quadratic", quad), ("quadratic, 5% noise", noisy), ("cubic", cubic), ("majority", maj), ("random", rand)]:
    F = (-1.0) ** f
    exact = 0.5 + 0.5 * U_pow(F, s + 1)
    print(f"{name:20s} accept prob (exact) {exact:.4f}   simulated {test_sim(f, s):.4f}")

For the acceptance probability: let $F=(-1)^f$. The test accepts when $\prod_\omega F(x+\omega\cdot h)=1$, so the probability of acceptance is $\mathbb E\frac{1+\prod_\omega F(x+\omega\cdot h)}2=\frac12+\frac12\|F\|_{U^{s+1}}^{2^{s+1}}$, by the cube formula over $\mathbb F_2^n$ (there is no conjugation since $F$ is real). A polynomial of degree at most $s$ has all its $(s+1)$-st derivatives zero, so it always passes, as the output shows for the quadratic.

The other functions pass with probability given exactly by their Gowers norms, and the simulation agrees to two decimals. Flipping $5\%$ of the values of the quadratic drops the acceptance to $0.71$: each of the $8$ points of the cube is corrupted with probability $0.05$, and $\frac12+\frac12(1-2\cdot0.05)^8=0.715$. The cubic passes with probability $0.67$: it agrees with a quadratic on a large part of the space, which the $U^3$ norm detects. Majority and random functions pass with probability close to $\frac12$, as functions far from every quadratic must.

This test is due to Alon, Kaufman, Krivelevich, Litsyn and Ron (2003), who showed that a function accepted with probability close to $1$ is close to a polynomial of degree at most $s$. The harder question, what a function accepted with probability only slightly above $\frac12$ looks like, is exactly the inverse theorem for the Gowers norms over finite fields, proved by Bergelson, Tao and Ziegler and by Tao and Ziegler in 2010–2012; it matters for decoding Reed–Muller codes far beyond their unique decoding radius.